In [4]:
# test_mlp.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from mlp import (MLP, build_mlp, forward_with_cache, softmax_cross_entropy,
                 SGDMomentum, n_params)


# =========================================================
# 1. KIẾN TRÚC & SHAPE
# =========================================================

def test_output_shape():
    model = build_mlp([784, 32, 10])
    X = torch.randn(16, 784)
    out, cache = forward_with_cache(model, X, train=False)
    assert out.shape == (16, 10), f"shape sai: {out.shape}"
    assert len(cache) == len(model), "cache phải có một phần tử cho mỗi layer"


def test_param_count():
    # 784*32 + 32  +  32*10 + 10 = 25450
    model = build_mlp([784, 32, 10])
    assert n_params(model) == 25450, n_params(model)


def test_invalid_args_raise():
    for bad in [dict(activation="gelu"), dict(init="unknown"), dict(p_drop=1.0)]:
        try:
            build_mlp([4, 3, 2], **bad)
        except ValueError:
            continue
        raise AssertionError(f"Không raise ValueError với {bad}")


# =========================================================
# 2. LOSS
# =========================================================

def test_softmax_ce_matches_pytorch():
    torch.manual_seed(0)
    logits = torch.randn(20, 5)
    y = torch.randint(0, 5, (20,))
    loss, delta = softmax_cross_entropy(logits, y)
    ref_loss = F.cross_entropy(logits, y)
    assert torch.allclose(loss, ref_loss, atol=1e-6), (loss, ref_loss)
    # delta phải bằng gradient của loss theo logits
    with torch.enable_grad():
        z = logits.clone().requires_grad_(True)
        F.cross_entropy(z, y).backward()
    assert torch.allclose(delta, z.grad, atol=1e-6), "delta không khớp autograd"


def test_softmax_ce_numerically_stable():
    logits = torch.tensor([[1000.0, 0.0], [0.0, 1000.0]])
    y = torch.tensor([0, 0])
    loss, delta = softmax_cross_entropy(logits, y)
    assert torch.isfinite(loss) and torch.isfinite(delta).all(), "tràn số"
    assert abs(loss.item() - 500.0) < 1e-3, loss.item()


# =========================================================
# 3. BACKWARD: SO VỚI AUTOGRAD (TEST QUAN TRỌNG NHẤT)
# =========================================================

def test_gradients_match_autograd():
    for activation in ["relu", "sigmoid", "tanh"]:
        for wd in [0.0, 0.1]:
            mlp = MLP([5, 7, 4, 3], activation=activation, init="xavier", seed=0)
            mlp.model.double()                       # float64 để so sánh chính xác
            X = torch.randn(8, 5, dtype=torch.float64)
            y = torch.randint(0, 3, (8,))

            # Gradient thủ công
            logits, cache = mlp.forward_with_cache(X, train=False)
            _, delta = softmax_cross_entropy(logits, y)
            mlp.backward(cache, delta, weight_decay=wd)
            manual = {n: p.grad.clone() for n, p in mlp.model.named_parameters()}

            # Gradient bằng autograd
            with torch.enable_grad():
                for p in mlp.parameters():
                    p.grad = None
                ref = F.cross_entropy(mlp.model(X), y)
                if wd > 0:  # L2 chỉ áp cho W: thêm (wd/2)*||W||^2
                    for name, p in mlp.model.named_parameters():
                        if name.endswith("weight"):
                            ref = ref + 0.5 * wd * (p ** 2).sum()
                ref.backward()

            for name, p in mlp.model.named_parameters():
                assert torch.allclose(manual[name], p.grad, atol=1e-10), \
                    f"gradient sai ở {name} (act={activation}, wd={wd})"


# =========================================================
# 4. OPTIMIZER
# =========================================================

def test_sgd_momentum_one_step_by_hand():
    p = torch.ones(3)
    p.grad = torch.ones(3)
    opt = SGDMomentum([p], lr=0.1, momentum=0.9)
    # Bước 1: v = 0.9*0 - 0.1*1 = -0.1 ; p = 1 - 0.1 = 0.9
    opt.step()
    assert torch.allclose(p, torch.full((3,), 0.9)), p
    # Bước 2: v = 0.9*(-0.1) - 0.1 = -0.19 ; p = 0.9 - 0.19 = 0.71
    opt.step()
    assert torch.allclose(p, torch.full((3,), 0.71)), p


# =========================================================
# 5. DROPOUT
# =========================================================

def test_dropout_eval_is_deterministic():
    model = build_mlp([10, 8, 3], p_drop=0.5)
    X = torch.randn(4, 10)
    o1, _ = forward_with_cache(model, X, train=False)
    o2, _ = forward_with_cache(model, X, train=False)
    assert torch.equal(o1, o2), "eval mode phải không ngẫu nhiên"


def test_inverted_dropout_preserves_mean():
    drop = nn.Sequential(nn.Dropout(0.5))
    X = torch.ones(100000, 1)
    out, _ = forward_with_cache(drop, X, train=True)
    assert abs(out.mean().item() - 1.0) < 0.02, out.mean().item()


# =========================================================
# 6. HUẤN LUYỆN END-TO-END
# =========================================================

def test_fit_reduces_loss():
    torch.manual_seed(1)
    X = torch.randn(128, 10)
    y = torch.randint(0, 3, (128,))
    mlp = MLP([10, 16, 3], seed=0)
    init_loss, _ = mlp.evaluate(X, y)
    hist = mlp.fit(X, y, epochs=30, batch_size=32, lr=0.1, verbose=False)
    assert hist["train_loss"][-1] < init_loss, "loss không giảm"


def test_learns_separable_data():
    torch.manual_seed(2)
    X = torch.randn(400, 2)
    y = (X[:, 0] + X[:, 1] > 0).long()           # bài toán tuyến tính, dễ học
    mlp = MLP([2, 16, 2], seed=0)
    mlp.fit(X, y, epochs=100, batch_size=32, lr=0.1, verbose=False)
    acc = (mlp.predict(X) == y).float().mean().item()
    assert acc > 0.95, f"độ chính xác thấp: {acc:.3f}"


def test_predict_shapes_and_probabilities():
    mlp = MLP([6, 8, 4], seed=0)
    X = torch.randn(10, 6)
    assert mlp.predict(X).shape == (10,)
    proba = mlp.predict_proba(X)
    assert proba.shape == (10, 4)
    assert torch.allclose(proba.sum(dim=1), torch.ones(10), atol=1e-6)


def test_fit_rejects_bad_input():
    mlp = MLP([4, 3, 2], seed=0)
    try:
        mlp.fit(torch.randn(5, 7), torch.zeros(5).long(), epochs=1, verbose=False)
    except ValueError:
        pass
    else:
        raise AssertionError("Không báo lỗi khi sai số feature")


# =========================================================
# CHẠY TRỰC TIẾP: python test_mlp.py
# =========================================================

if __name__ == "__main__":
    tests = [(name, fn) for name, fn in list(globals().items())
             if name.startswith("test_") and callable(fn)]
    passed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"[PASS] {name}")
            passed += 1
        except AssertionError as e:
            print(f"[FAIL] {name}: {e}")
    print(f"\n{passed}/{len(tests)} test đạt.")

[PASS] test_output_shape
[PASS] test_param_count
[PASS] test_invalid_args_raise
[PASS] test_softmax_ce_matches_pytorch
[PASS] test_softmax_ce_numerically_stable
[PASS] test_gradients_match_autograd
[PASS] test_sgd_momentum_one_step_by_hand
[PASS] test_dropout_eval_is_deterministic
[PASS] test_inverted_dropout_preserves_mean
[PASS] test_fit_reduces_loss
[PASS] test_learns_separable_data
[PASS] test_predict_shapes_and_probabilities
[PASS] test_fit_rejects_bad_input

13/13 test đạt.


In [5]:
# test_logistic_regression.py
import numpy as np

from logistic_regression import LogisticRegression, add_bias, softmax, cross_entropy

N_CLASSES = 10  # class LogisticRegression đang hardcode n_classes = 10


# =========================================================
# HÀM THAM CHIẾU ĐỘC LẬP (viết lại để so sánh)
# =========================================================

def naive_log_softmax(Z):
    e = np.exp(Z)
    return np.log(e / e.sum(axis=1, keepdims=True))


def full_loss(W, Xb, y, lam):
    """Loss đầy đủ mà code đang báo cáo: CE trung bình + 0.5*lam*||W_reg||^2 (không phạt bias)."""
    logp = naive_log_softmax(Xb @ W)
    ce = -np.mean(logp[np.arange(len(y)), y])
    W_reg = W.copy()
    W_reg[-1, :] = 0.0
    return ce + 0.5 * lam * np.sum(W_reg ** 2)


def numerical_grad(f, W, eps=1e-6):
    G = np.zeros_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            E = np.zeros_like(W)
            E[i, j] = eps
            G[i, j] = (f(W + E) - f(W - E)) / (2 * eps)
    return G


# =========================================================
# 1. TIỆN ÍCH: add_bias, softmax, cross_entropy
# =========================================================

def test_add_bias_shape_and_values():
    X = np.arange(6, dtype=float).reshape(3, 2)
    Xb = add_bias(X)
    assert Xb.shape == (3, 3), Xb.shape
    assert np.allclose(Xb[:, :2], X), "phần feature bị thay đổi"
    assert np.all(Xb[:, -1] == 1.0), "cột bias phải toàn 1"


def test_softmax_rows_sum_to_one():
    rng = np.random.default_rng(0)
    P = softmax(rng.normal(size=(20, 5)))
    assert np.allclose(P.sum(axis=1), 1.0, atol=1e-12)
    assert np.all(P >= 0)


def test_softmax_numerically_stable():
    Z = np.array([[1000.0, 0.0, -1000.0], [-1e4, 1e4, 0.0]])
    P = softmax(Z)
    assert np.all(np.isfinite(P)), "softmax tràn số"
    assert np.allclose(P[0], [1.0, 0.0, 0.0], atol=1e-12)
    assert np.allclose(P[1], [0.0, 1.0, 0.0], atol=1e-12)


def test_softmax_matches_naive():
    rng = np.random.default_rng(1)
    Z = rng.normal(size=(7, 4))
    naive = np.exp(Z) / np.exp(Z).sum(axis=1, keepdims=True)
    assert np.allclose(softmax(Z), naive, atol=1e-12)


def test_cross_entropy_matches_naive():
    rng = np.random.default_rng(2)
    Z = rng.normal(size=(15, N_CLASSES))
    y = rng.integers(0, N_CLASSES, 15)
    logp = naive_log_softmax(Z)
    ref = -np.mean(logp[np.arange(15), y])
    assert np.isclose(cross_entropy(Z, y), ref, atol=1e-10), (cross_entropy(Z, y), ref)


def test_cross_entropy_stable_and_correct_on_extreme_logits():
    Z = np.array([[1000.0, 0.0], [0.0, 1000.0]])
    y = np.array([0, 0])
    loss = cross_entropy(Z, y)
    assert np.isfinite(loss), "cross_entropy tràn số"
    assert abs(loss - 500.0) < 1e-6, loss   # mẫu thứ 2 có lỗi 1000, trung bình = 500


def test_cross_entropy_uniform_logits_equals_log_k():
    Z = np.zeros((10, N_CLASSES))
    y = np.arange(10)
    assert np.isclose(cross_entropy(Z, y), np.log(N_CLASSES)), cross_entropy(Z, y)


# =========================================================
# 2. GRADIENT: SO VỚI SỐ HỌC (TEST QUAN TRỌNG NHẤT)
# =========================================================

def test_analytic_gradient_matches_numerical():
    """Công thức code dùng: X^T (P - Y) / N + lam * mask * W phải là đạo hàm của full_loss."""
    rng = np.random.default_rng(3)
    X = rng.normal(size=(12, 4))
    y = rng.integers(0, N_CLASSES, 12)
    lam = 0.1
    Xb = add_bias(X)
    N, d = Xb.shape
    W = rng.normal(size=(d, N_CLASSES)) * 0.5

    P = softmax(Xb @ W)
    Y = np.zeros_like(P)
    Y[np.arange(N), y] = 1.0
    mask = np.ones((d, N_CLASSES))
    mask[-1, :] = 0.0
    analytic = Xb.T @ (P - Y) / N + lam * mask * W

    numeric = numerical_grad(lambda W_: full_loss(W_, Xb, y, lam), W)
    assert np.allclose(analytic, numeric, atol=1e-6), np.max(np.abs(analytic - numeric))


def test_one_full_batch_step_matches_theory():
    """Chạy đúng 1 bước full-batch từ W = 0 rồi so với W = -eta * grad số học."""
    rng = np.random.default_rng(4)
    X = rng.normal(size=(30, 4))
    y = rng.integers(0, N_CLASSES, 30)
    eta, lam = 0.1, 0.05

    model = LogisticRegression(eta=eta, lam=lam, max_epochs=1,
                               batch_size=len(X)).fit(X, y)

    Xb = add_bias(X)
    W0 = np.zeros((Xb.shape[1], N_CLASSES))
    g = numerical_grad(lambda W_: full_loss(W_, Xb, y, lam), W0)
    assert np.allclose(model.W_, W0 - eta * g, atol=1e-6), \
        "cập nhật trọng số không khớp -eta * gradient"


def test_bias_row_not_regularized():
    """Với lam lớn, hàng bias không được co về 0 do phạt L2 (chỉ phạt W, không phạt bias)."""
    rng = np.random.default_rng(5)
    X = rng.normal(size=(20, 3))
    y = rng.integers(0, N_CLASSES, 20)
    lam = 100.0
    model = LogisticRegression(eta=0.01, lam=lam, max_epochs=1,
                               batch_size=len(X)).fit(X, y)

    Xb = add_bias(X)
    W0 = np.zeros((Xb.shape[1], N_CLASSES))
    # Gradient của bias chỉ đến từ CE, không có lam
    P = softmax(Xb @ W0)
    Y = np.zeros_like(P)
    Y[np.arange(len(y)), y] = 1.0
    bias_grad = (Xb.T @ (P - Y) / len(y))[-1]
    assert np.allclose(model.W_[-1], -0.01 * bias_grad, atol=1e-8), \
        "hàng bias đang bị phạt L2 sai"


# =========================================================
# 3. HUẤN LUYỆN END-TO-END
# =========================================================

def make_blobs(n_per_class=50, std=0.3, radius=5.0, seed=0):
    """10 cụm đặt trên đường tròn: mỗi cụm tách tuyến tính được khỏi phần còn lại."""
    rng = np.random.default_rng(seed)
    angles = 2 * np.pi * np.arange(N_CLASSES) / N_CLASSES
    centers = radius * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    X = np.vstack([c + std * rng.normal(size=(n_per_class, 2)) for c in centers])
    y = np.repeat(np.arange(N_CLASSES), n_per_class)
    return X, y


def test_learns_separable_blobs():
    X, y = make_blobs()
    model = LogisticRegression(eta=0.5, lam=0.0, max_epochs=200,
                               batch_size=64, seed=0).fit(X, y)
    acc = np.mean(model.predict(X) == y)
    assert acc > 0.95, f"độ chính xác thấp: {acc:.3f}"


def test_training_loss_decreases():
    X, y = make_blobs(seed=1)
    model = LogisticRegression(eta=0.3, lam=1e-3, max_epochs=50,
                               batch_size=64, seed=0).fit(X, y)
    h = model.history_["train_loss"]
    assert len(h) > 1
    assert h[-1] < h[0], f"loss không giảm: {h[0]:.4f} -> {h[-1]:.4f}"


def test_val_loss_recorded_each_epoch():
    X, y = make_blobs(seed=2)
    Xtr, ytr, Xva, yva = X[::2], y[::2], X[1::2], y[1::2]
    model = LogisticRegression(eta=0.3, max_epochs=10, batch_size=32, seed=0)
    model.fit(Xtr, ytr, X_val=Xva, y_val=yva)
    assert len(model.history_["val_loss"]) == len(model.history_["train_loss"])


def test_early_stopping_by_tol():
    X, y = make_blobs(seed=3)
    # tol rất lớn: sau epoch thứ 2 đã thỏa điều kiện dừng
    model = LogisticRegression(eta=0.1, max_epochs=100, batch_size=64,
                               tol=1e9, seed=0).fit(X, y)
    assert len(model.history_["train_loss"]) == 2, len(model.history_["train_loss"])


def test_same_seed_same_weights():
    X, y = make_blobs(seed=4)
    m1 = LogisticRegression(max_epochs=5, seed=7).fit(X, y)
    m2 = LogisticRegression(max_epochs=5, seed=7).fit(X, y)
    assert np.array_equal(m1.W_, m2.W_), "cùng seed phải cho cùng kết quả"


# =========================================================
# 4. DỰ ĐOÁN
# =========================================================

def test_predict_shapes_and_probabilities():
    X, y = make_blobs(n_per_class=5, seed=5)
    model = LogisticRegression(max_epochs=3, seed=0).fit(X, y)
    assert model.decision_function(X).shape == (len(X), N_CLASSES)
    P = model.predict_proba(X)
    assert P.shape == (len(X), N_CLASSES)
    assert np.allclose(P.sum(axis=1), 1.0, atol=1e-12)
    pred = model.predict(X)
    assert pred.shape == (len(X),)
    assert set(np.unique(pred)).issubset(set(range(N_CLASSES)))


def test_predict_consistent_with_proba():
    X, y = make_blobs(n_per_class=5, seed=6)
    model = LogisticRegression(max_epochs=3, seed=0).fit(X, y)
    assert np.array_equal(model.predict(X), np.argmax(model.predict_proba(X), axis=1))


# =========================================================
# CHẠY TRỰC TIẾP: python test_logistic_regression.py
# =========================================================

if __name__ == "__main__":
    tests = [(name, fn) for name, fn in list(globals().items())
             if name.startswith("test_") and callable(fn)]
    passed = 0
    for name, fn in tests:
        try:
            fn()
            print(f"[PASS] {name}")
            passed += 1
        except AssertionError as e:
            print(f"[FAIL] {name}: {e}")
    print(f"\n{passed}/{len(tests)} test đạt.")

[PASS] test_output_shape
[PASS] test_param_count
[PASS] test_invalid_args_raise
[PASS] test_softmax_ce_matches_pytorch
[PASS] test_softmax_ce_numerically_stable
[PASS] test_gradients_match_autograd
[PASS] test_sgd_momentum_one_step_by_hand
[PASS] test_dropout_eval_is_deterministic
[PASS] test_inverted_dropout_preserves_mean
[PASS] test_fit_reduces_loss
[PASS] test_learns_separable_data
[PASS] test_predict_shapes_and_probabilities
[PASS] test_fit_rejects_bad_input
[PASS] test_add_bias_shape_and_values
[PASS] test_softmax_rows_sum_to_one
[PASS] test_softmax_numerically_stable
[PASS] test_softmax_matches_naive
[PASS] test_cross_entropy_matches_naive
[PASS] test_cross_entropy_stable_and_correct_on_extreme_logits
[PASS] test_cross_entropy_uniform_logits_equals_log_k
[PASS] test_analytic_gradient_matches_numerical
[PASS] test_one_full_batch_step_matches_theory
[PASS] test_bias_row_not_regularized
[PASS] test_learns_separable_blobs
[PASS] test_training_loss_decreases
[PASS] test_val_loss_reco